In [ ]:
#The code reads a CSV file into a Pandas DataFrame and previews the dataset contents.

import pandas as pd

df = pd.read_csv("../data/email_evaluation_dataset_misa.csv")
df.head()


,id,email_text,expected_action,expected_tone
0,1,Reminder: This is a gentle reminder about the ...,notify,neutral
1,2,Reminder: This is a gentle reminder about the ...,notify,neutral
2,3,Reminder: This is a gentle reminder about the ...,notify,neutral
3,4,Reminder: This is a gentle reminder about the ...,notify,neutral
4,5,Reminder: This is a gentle reminder about the ...,notify,neutral


In [ ]:
#This text convert the email text to lowercase, removes non-alphabetic characters, and trims whitespace.

source_col = 'email_text'

df['clean_text'] = (
    df[source_col]
      .fillna('')
      .astype(str)
      .str.lower()
      .str.replace(r'[^a-z\s]', '', regex=True)
      .str.strip()
)

df[[source_col, 'clean_text']].head()


,email_text,clean_text
0,Reminder: This is a gentle reminder about the ...,reminder this is a gentle reminder about the p...
1,Reminder: This is a gentle reminder about the ...,reminder this is a gentle reminder about the p...
2,Reminder: This is a gentle reminder about the ...,reminder this is a gentle reminder about the p...
3,Reminder: This is a gentle reminder about the ...,reminder this is a gentle reminder about the p...
4,Reminder: This is a gentle reminder about the ...,reminder this is a gentle reminder about the p...


In [ ]:
#This text removes stop words from the cleaned email text and extracts keywords.

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words("english"))

df['keywords'] = df['clean_text'].apply(
    lambda x: [w for w in x.split() if w not in stop_words]
)

df[['clean_text', 'keywords']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\barat\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,clean_text,keywords
0,reminder this is a gentle reminder about the p...,"[reminder, gentle, reminder, project, review, ..."
1,reminder this is a gentle reminder about the p...,"[reminder, gentle, reminder, project, review, ..."
2,reminder this is a gentle reminder about the p...,"[reminder, gentle, reminder, project, review, ..."
3,reminder this is a gentle reminder about the p...,"[reminder, gentle, reminder, project, review, ..."
4,reminder this is a gentle reminder about the p...,"[reminder, gentle, reminder, project, review, ..."


In [ ]:
#This text defines a triage rule function that categorizes email text into "respond", "notify", or "ignore" based on specific keywords.

def triage_rule(text):
    text = text.lower()

    # URGENT / IMPORTANT cases
    if any(word in text for word in [
        "urgent", "invoice", "payment", "due", "password",
        "account", "verification", "deadline"
    ]):
        return "respond"

    # NOTIFICATION cases
    if any(word in text for word in [
        "meeting", "reminder", "schedule", "webinar"
    ]):
        return "notify"

    # NO ACTION REQUIRED cases
    if any(word in text for word in [
        "no action required", "for your reference",
        "newsletter", "promotion", "thank you"
    ]):
        return "ignore"

    # DEFAULT
    return "respond"


In [ ]:
#This text applies the triage rule function to the cleaned email text and displays the results.

df['assistant_action'] = df['clean_text'].apply(triage_rule)

df[['email_text', 'assistant_action']].head()


,email_text,assistant_action
0,Reminder: This is a gentle reminder about the ...,notify
1,Reminder: This is a gentle reminder about the ...,notify
2,Reminder: This is a gentle reminder about the ...,notify
3,Reminder: This is a gentle reminder about the ...,notify
4,Reminder: This is a gentle reminder about the ...,notify


In [ ]:
#This code evaluates the assistant's actions against expected actions and shows the results.

df['correct_prediction'] = (
    df['assistant_action'] == df['expected_action']
)

df[['email_text', 'expected_action', 'assistant_action', 'correct_prediction']].head()


,email_text,expected_action,assistant_action,correct_prediction
0,Reminder: This is a gentle reminder about the ...,notify,notify,True
1,Reminder: This is a gentle reminder about the ...,notify,notify,True
2,Reminder: This is a gentle reminder about the ...,notify,notify,True
3,Reminder: This is a gentle reminder about the ...,notify,notify,True
4,Reminder: This is a gentle reminder about the ...,notify,notify,True


In [ ]:
#This code calculates and prints the accuracy of the assistant's predictions.

accuracy = df['correct_prediction'].mean()
print(f"Assistant Accuracy: {accuracy:.2%}")


Assistant Accuracy: 80.00%


In [ ]:
#This code saves the updated DataFrame with assistant actions and correctness evaluations to a new CSV file.

df.to_csv("../data/milestone2_output_MisaKanaujiya.csv", index=False)

In [ ]:
#This code applies the triage rule function to the cleaned email text and displays the results.

# Check columns
print(df.columns)

# Apply triage rule
df['predicted_triage'] = df['clean_text'].apply(triage_rule)

# View result
df[['email_text', 'predicted_triage']].head()


Index(['id', 'email_text', 'expected_action', 'expected_tone', 'clean_text',
       'ideal_response', 'assistant_action', 'correct_prediction'],
      dtype='object')


,email_text,predicted_triage
0,Reminder: This is a gentle reminder about the ...,notify
1,Reminder: This is a gentle reminder about the ...,notify
2,Reminder: This is a gentle reminder about the ...,notify
3,Reminder: This is a gentle reminder about the ...,notify
4,Reminder: This is a gentle reminder about the ...,notify


In [ ]:
#This text defines a function that determines the ideal assistant response based on email content.

def ideal_response(text):
    """
    Returns the ideal assistant action based on email content.
    Possible outputs: respond, notify, ignore
    """
    text = text.lower()

    # Immediate or critical action required
    if any(word in text for word in [
        "urgent", "invoice", "payment", "due",
        "password", "account", "verification", "deadline"
    ]):
        return "respond"

    # Informational notifications
    if any(word in text for word in [
        "meeting", "reminder", "schedule",
        "webinar", "session", "agenda"
    ]):
        return "notify"

    # No action required
    if any(word in text for word in [
        "no action required", "for your reference",
        "newsletter", "promotion", "thank you",
        "fyi"
    ]):
        return "ignore"

    # Default safe behavior
    return "respond"


In [ ]:
#This code applies the ideal_response function to the cleaned email text to generate the ideal action for 
#each email and then displays a preview of the email content along with the corresponding ideal action.

df['ideal_action'] = df['clean_text'].apply(ideal_response)

df[['email_text', 'ideal_action']].head()


,email_text,ideal_action
0,Reminder: This is a gentle reminder about the ...,notify
1,Reminder: This is a gentle reminder about the ...,notify
2,Reminder: This is a gentle reminder about the ...,notify
3,Reminder: This is a gentle reminder about the ...,notify
4,Reminder: This is a gentle reminder about the ...,notify


Reflection
1. Which emails were most difficult to handle?

Emails that did not clearly fall into a single category were the most challenging to classify. These included updates, announcements, or informational messages that appeared important but did not always demand immediate action. Some promotional emails were also misleading because they mixed useful information, such as event details, with marketing content, making it unclear whether the user needed to respond or simply read them.

2. What caused errors in the rule-based approach?

The rule-based system produced errors because it depended heavily on keyword matching rather than true understanding. Certain keywords triggered responses even when no action was required, while other emails lacked strong keywords but still carried urgency. The rules were also limited in identifying emotional tone, as they could not interpret subtle language or sentence structure.

3. How can a large language model improve performance?

A large language model can improve accuracy by interpreting the overall meaning and purpose of an email instead of focusing on isolated words. It can better judge whether an email requires action, notification, or no response at all. Additionally, an LLM can recognize nuanced expressions of urgency or politeness, leading to more reliable decisions in complex or ambiguous scenarios.

In [4]:
!pip install langsmith


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import langsmith as ls
client = ls.Client()


In [6]:
#Define the Judge Prompt

judge_prompt = """You are an evaluator.Compare the model output with the ideal answer.

check:
1. Action Correctness: Did the model choose the correct action (respond, notify, ignore) based on the email content?
2. Tone Correctness: Did the model use an appropriate tone (formal, informal, neutral) for the email context?

Give Score:
1 = correct
0 = incorrect """

In [7]:
# Run agent + Judge

def evaluate(agent_output, ideal_action, ideal_tone):
    if (
        agent_output["action"] == ideal_action and
        agent_output["tone"] == ideal_tone
    ):
        return 1
    else:
        return 0


In [8]:
agent_output = {
    "action" : "notify",
    "tone" : "urgent"}

In [9]:
ideal_action = "notify"
ideal_tone = "urgent"

In [10]:
score = evaluate(agent_output, ideal_action, ideal_tone)
print(f"Evaluation Score: {score}")

Evaluation Score: 1


In [12]:
import pandas as pd

df = pd.read_csv("../data/email_evaluation_dataset_misa.csv")
df.head()

,id,email_text,expected_action,expected_tone
0,1,Reminder: This is a gentle reminder about the ...,notify,neutral
1,2,Reminder: This is a gentle reminder about the ...,notify,neutral
2,3,Reminder: This is a gentle reminder about the ...,notify,neutral
3,4,Reminder: This is a gentle reminder about the ...,notify,neutral
4,5,Reminder: This is a gentle reminder about the ...,notify,neutral


In [23]:
def email_assistant(email_text):
    text = email_text.lower()

    if "urgent" in text or "invoice" in text or "payment" in text or "due" in text:
        return {"action": "respond", "tone": "urgent"}

    elif "meeting" in text or "reminder" in text or "schedule" in text:
        return {"action": "notify", "tone": "neutral"}

    elif "thank you" in text or "thanks" in text or "no action required" in text:
        return {"action": "ignore", "tone": "neutral"}

    else:
        return {"action": "respond", "tone": "polite"}


In [24]:
# Run the evaluation for the sample dataset

scores = []
for index, row in df.iterrows():
    prediction = email_assistant(row['email_text'])
    score = evaluate(
        prediction,
        row['expected_action'],
        row['expected_tone']
    )
    scores.append(score)

accuracy = sum(scores) / len(scores) if scores else 0.0
print(f"Overall Evaluation Accuracy: {accuracy:.2%}")

Overall Evaluation Accuracy: 100.00%
